---
## Section 0 — Authentication, Hardware & Drive Mount

In [ ]:
# Fix pyarrow version
!pip uninstall -y pyarrow
!pip install pyarrow==14.0.1
!pip install --upgrade datasets

# Now import everything
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import drive
drive.mount('/content/drive')

# Then continue with your CFG, model loading, etc.

Found existing installation: pyarrow 18.1.0
Uninstalling pyarrow-18.1.0:
  Successfully uninstalled pyarrow-18.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 71.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires pyarrow>=15.0.0, but you have pyarrow 14.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pyarrow>=15.0.0; platform_machine == "x86_64", but you have pyarrow 14.0.1 which is incompatible.
bigframes 2.39.0 requires pyarrow>=15.0.2, but you have pyarrow 14.0.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.1
    Uninstalling pyarrow-14.0.1:
      Successfully uninstalled pyarrow-14.0.1
  

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# ────────────────────────────────────────────────────────────
# 0.1  Hugging Face authentication  ← FIXES the 401 error
# ────────────────────────────────────────────────────────────
import os
from google.colab import userdata

# Try Colab Secrets first (recommended)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        os.environ['HF_TOKEN'] = HF_TOKEN
        print('✅ HF_TOKEN loaded from Colab Secrets.')
    else:
        raise ValueError('empty')
except Exception:
    print('⚠️  HF_TOKEN not in Colab Secrets.')
    print('   Option A (recommended): Runtime → Manage Secrets → add HF_TOKEN')
    print('   Option B: paste your token below (never commit this!):')
    HF_TOKEN = ''   # ← paste token here only if using Option B
    if HF_TOKEN:
        os.environ['HF_TOKEN'] = HF_TOKEN
        print('   Token set from inline variable.')
    else:
        print('   Continuing without token — only public models will work.')

# Login to HF Hub so all library calls inherit the token
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Logged in to Hugging Face Hub.')

✅ HF_TOKEN loaded from Colab Secrets.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in to Hugging Face Hub.


In [ ]:
# ────────────────────────────────────────────────────────────
# 0.2  Hardware diagnostics
# ────────────────────────────────────────────────────────────
import subprocess, platform, torch

print('=' * 60)
print('HARDWARE REPORT')
print('=' * 60)
try:
    print(subprocess.check_output(['nvidia-smi'], encoding='utf-8'))
except Exception:
    print('No NVIDIA GPU found.')

!cat /proc/cpuinfo | grep 'model name' | head -1
!free -h
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.version.cuda)
print('Device  :', 'cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

HARDWARE REPORT
Wed Apr 22 09:10:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-------------------------------

In [ ]:
# ────────────────────────────────────────────────────────────
# 0.3  Mount Google Drive
# ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/MAPO_Reproduction_Version2'
for d in [
    f'{DRIVE_ROOT}/datasets',
    f'{DRIVE_ROOT}/models/dpo_iter1',
    f'{DRIVE_ROOT}/models/dpo_iter2',
    f'{DRIVE_ROOT}/models/dpo_iter3',
    f'{DRIVE_ROOT}/models/ppo_lora',
    f'{DRIVE_ROOT}/preference_data',
    f'{DRIVE_ROOT}/results',
    f'{DRIVE_ROOT}/logs',
    f'{DRIVE_ROOT}/figures',
]:
    os.makedirs(d, exist_ok=True)
print('✅ Drive mounted and folder tree created at:', DRIVE_ROOT)

Mounted at /content/drive
✅ Drive mounted and folder tree created at: /content/drive/MyDrive/MAPO_Reproduction_Version2


---
## Section 1 — Install Packages & Clone MAPO Repo

In [ ]:
# Install all required packages
!pip install -q \
    'transformers>=4.40.0' \
    'datasets>=2.19.0' \
    'accelerate>=0.30.0' \
    'peft>=0.10.0' \
    'trl>=0.8.6' \
    'bitsandbytes>=0.43.1' \
    sentencepiece \
    sacrebleu evaluate \
    scipy matplotlib seaborn pandas numpy \
    huggingface_hub \
    requests

# Clone official MAPO repo
if not os.path.exists('/content/MAPO'):
    !git clone https://github.com/NJUNLP/MAPO /content/MAPO
else:
    !cd /content/MAPO && git pull --quiet
%cd /content/MAPO
!ls

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.2 MB/s eta 0:00:00
Cloning into '/content/MAPO'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 207 (delta 100), reused 170 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (207/207), 35.64 MiB | 16.08 MiB/s, done.
Resolving deltas: 100% (100/100), done.
Updating files: 100% (51/51), done.
/content/MAPO
Alignment  Data  Evaluation  fig  Preprocess  README.md  requirement.txt


In [ ]:

import torch

BASE_MODEL_PREFERRED  = 'kevinpro/MathOctopus-MAPO-DPO-7B'
BASE_MODEL_FALLBACK_A = 'mistralai/Mistral-7B-v0.1'
BASE_MODEL_FALLBACK_B = 'meta-llama/Llama-2-7b-hf'

def probe_model(repo_id):
    """Return True if we can download config.json for this model."""
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download(repo_id=repo_id, filename='config.json',
                        token=os.environ.get('HF_TOKEN'))
        return True
    except Exception as e:
        print(f'  {repo_id}: {str(e)[:80]}')
        return False

print('Probing model availability...')
if probe_model(BASE_MODEL_PREFERRED):
    CHOSEN_MODEL = BASE_MODEL_PREFERRED
    MODEL_SHORT  = 'MathOctopus-7B'
elif probe_model(BASE_MODEL_FALLBACK_A):
    CHOSEN_MODEL = BASE_MODEL_FALLBACK_A
    MODEL_SHORT  = 'Mistral-7B-v0.1-fallback'
    print('⚠️  Using Mistral-7B as fallback. Fine-tune on MGSM8KInstruct first for best results.')
else:
    CHOSEN_MODEL = BASE_MODEL_FALLBACK_B
    MODEL_SHORT  = 'Llama-2-7B-fallback'
    print('⚠️  Using Llama-2-7B as fallback. Ensure Meta license accepted.')

print(f'\n✅ Using model: {CHOSEN_MODEL}')

CFG = {
    'base_model':      CHOSEN_MODEL,
    'model_short':     MODEL_SHORT,
    'nllb_model':      'facebook/nllb-200-distilled-600M',
    'n_samples':       8,
    'max_new_tokens':  512,
    'temperature':     0.7,
    # DPO
    'dpo_lr':          1e-6,
    'dpo_beta':        0.1,
    'dpo_warmup':      100,
    'dpo_batch_size':  128,
    'dpo_max_steps':   1000,
    'dpo_iterations':  3,
    # PPO-LoRA
    'ppo_lr':          2e-5,
    'ppo_batch_size':  64,
    'ppo_epochs':      2,
    'ppo_max_steps':   2600,
    'ppo_warmup':      150,
    'lora_r':          128,
    'lora_alpha':      256,
    'lora_target':     ['q_proj', 'v_proj', 'o_proj'],
    # Paths
    'drive_root':      DRIVE_ROOT,
    'output_dir':      f'{DRIVE_ROOT}/models',
    'results_dir':     f'{DRIVE_ROOT}/results',
    'logs_dir':        f'{DRIVE_ROOT}/logs',
    'figures_dir':     f'{DRIVE_ROOT}/figures',
    # Runtime
    'device':   'cuda' if torch.cuda.is_available() else 'cpu',
    'bf16': torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
    'fp16': torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8,
    # Languages
    'languages':  ['bn','th','sw','ja','zh','ru','de','es','fr','en'],
    'lang_names': {
        'bn':'Bengali','th':'Thai','sw':'Swahili','ja':'Japanese',
        'zh':'Chinese','ru':'Russian','de':'German',
        'es':'Spanish','fr':'French','en':'English'
    },
    'nllb_codes': {
        'bn':'ben_Beng','th':'tha_Thai','sw':'swh_Latn','ja':'jpn_Jpan',
        'zh':'zho_Hans','ru':'rus_Cyrl','de':'deu_Latn',
        'es':'spa_Latn','fr':'fra_Latn','en':'eng_Latn'
    },
}
print('Configuration ready.')

Probing model availability...


config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]


✅ Using model: kevinpro/MathOctopus-MAPO-DPO-7B
Configuration ready.


---
## Section 2 — Dataset Acquisition

In [ ]:

import json
import requests
import csv
import io
from collections import defaultdict

LANGUAGES = CFG['languages']

MGSM_TSV_BASE = 'https://raw.githubusercontent.com/google-research/url-nlp/main/mgsm/mgsm_{lang}.tsv'

MGSM_JSON_BASE = 'https://raw.githubusercontent.com/ruimashita/MGSM/main/mgsm_{lang}.json'

def download_mgsm_from_github(lang: str):
    """Download MGSM test TSV or JSON from official GitHub repos."""

    tsv_url = MGSM_TSV_BASE.format(lang=lang)
    try:
        r = requests.get(tsv_url, timeout=15)
        if r.status_code == 200:
            reader = csv.reader(io.StringIO(r.text), delimiter='\t')
            rows = list(reader)
            # Skip header if present
            if rows and rows[0][0].lower() in ('question', 'input'):
                rows = rows[1:]
            return [{'question': row[0], 'answer': row[1]} for row in rows if len(row) >= 2]
    except Exception:
        pass

    # Try JSON fallback
    json_url = MGSM_JSON_BASE.format(lang=lang)
    try:
        r = requests.get(json_url, timeout=15)
        if r.status_code == 200:
            data = r.json()
            if isinstance(data, list):
                return [{'question': d.get('question', ''), 'answer': str(d.get('answer', ''))} for d in data]
    except Exception:
        pass
    return None

def build_synthetic_mgsm_fallback(lang: str, n: int = 50):
    """Minimal synthetic placeholder so pipeline doesn't crash."""
    SAMPLE_PROBLEMS = [
        ('James has 3 apples. He buys 5 more. How many apples does he have?', '8'),
        ('A store has 24 shirts. They sell 9. How many shirts remain?', '15'),
        ('Maria earns $12 per hour. She works 8 hours. How much does she earn?', '96'),
        ('There are 4 boxes with 6 oranges each. How many oranges total?', '24'),
        ('John has 50 coins. He spends 18. How many are left?', '32'),
    ]
    items = []
    for i in range(n):
        q, a = SAMPLE_PROBLEMS[i % len(SAMPLE_PROBLEMS)]
        items.append({'question': f'[{lang}] {q}', 'answer': a, 'synthetic': True})
    return items

print('Loading MGSM dataset...')
mgsm_data = {}
for lang in LANGUAGES:
    result = download_mgsm_from_github(lang)
    if result:
        mgsm_data[lang] = result
        print(f'  ✅ MGSM [{lang}] from GitHub: {len(result)} examples')
    else:
        # Synthetic fallback – pipeline will still run, but real evaluation impossible
        mgsm_data[lang] = build_synthetic_mgsm_fallback(lang, n=50)
        print(f'  ⚠️  MGSM [{lang}] synthetic fallback (50 samples) — real evals not possible')

# Save to Drive
with open(f'{DRIVE_ROOT}/datasets/mgsm.json', 'w') as f:
    json.dump(mgsm_data, f, ensure_ascii=False, indent=2)
print('MGSM saved to Drive.')

Loading MGSM dataset...
  ✅ MGSM [bn] from GitHub: 250 examples
  ✅ MGSM [th] from GitHub: 250 examples
  ✅ MGSM [sw] from GitHub: 250 examples
  ✅ MGSM [ja] from GitHub: 250 examples
  ✅ MGSM [zh] from GitHub: 250 examples
  ✅ MGSM [ru] from GitHub: 250 examples
  ✅ MGSM [de] from GitHub: 250 examples
  ✅ MGSM [es] from GitHub: 250 examples
  ✅ MGSM [fr] from GitHub: 250 examples
  ✅ MGSM [en] from GitHub: 250 examples
MGSM saved to Drive.


In [ ]:

import pandas as pd
import requests
import json
import os

LANGUAGES = CFG['languages']

MSVAMP_BASE_URL = "https://huggingface.co/datasets/Mathoctopus/MSVAMP/resolve/main/"
MSVAMP_FILES = {
    'bn': 'test_Bengali.json',
    'th': 'test_Thai.json',
    'sw': 'test_Swahili.json',
    'ja': 'test_Japanese.json',
    'zh': 'test_Chinese.json',
    'ru': 'test_Russian.json',
    'de': 'test_German.json',
    'es': 'test_Spanish.json',
    'fr': 'test_French.json',
    'en': 'test_English.json',
}

def load_msvamp_pandas(lang: str):
    """Load MSVAMP JSON Lines file using pandas."""
    filename = MSVAMP_FILES.get(lang)
    if not filename:
        return None
    url = MSVAMP_BASE_URL + filename
    try:

        df = pd.read_json(url, lines=True)

        if 'Body' in df.columns and 'Answer' in df.columns:
            return [
                {"question": row['Body'], "answer": str(row['Answer'])}
                for _, row in df.iterrows()
            ]
        else:

            q_col = df.columns[0]
            a_col = df.columns[-1]
            return [
                {"question": row[q_col], "answer": str(row[a_col])}
                for _, row in df.iterrows()
            ]
    except Exception as e:
        print(f"    Pandas load failed for {lang}: {str(e)[:80]}")
        return None

def build_synthetic_msvamp_fallback(lang: str, n: int = 50):
    """Minimal synthetic placeholder so pipeline doesn't crash."""
    return [
        {"question": f"Synthetic MSVAMP problem {i} in {lang}", "answer": str(i % 100), "synthetic": True}
        for i in range(n)
    ]

print("Loading MSVAMP dataset...")
msvamp_data = {}

for lang in LANGUAGES:
    result = load_msvamp_pandas(lang)
    if result:
        msvamp_data[lang] = result
        print(f"  ✅ MSVAMP [{lang}] from Hugging Face (pandas): {len(result)} examples")
    else:
        msvamp_data[lang] = build_synthetic_msvamp_fallback(lang, n=50)
        print(f"  ⚠️  MSVAMP [{lang}] synthetic fallback (50 samples) — real evals not possible")

# Save to Drive
os.makedirs(f"{DRIVE_ROOT}/datasets", exist_ok=True)
with open(f"{DRIVE_ROOT}/datasets/msvamp.json", "w") as f:
    json.dump(msvamp_data, f, ensure_ascii=False, indent=2)
print("MSVAMP saved to Drive.")

Loading MSVAMP dataset...
  ✅ MSVAMP [bn] from Hugging Face (pandas): 1000 examples
  ✅ MSVAMP [th] from Hugging Face (pandas): 1000 examples
  ✅ MSVAMP [sw] from Hugging Face (pandas): 1000 examples
  ✅ MSVAMP [ja] from Hugging Face (pandas): 1000 examples
  ✅ MSVAMP [zh] from Hugging Face (pandas): 1000 examples
  ✅ MSVAMP [ru] from Hugging Face (pandas): 1000 examples
  ✅ MSVAMP [de] from Hugging Face (pandas): 1000 examples
  ✅ MSVAMP [es] from Hugging Face (pandas): 1000 examples
  ✅ MSVAMP [fr] from Hugging Face (pandas): 1000 examples
  ✅ MSVAMP [en] from Hugging Face (pandas): 1000 examples
MSVAMP saved to Drive.


In [ ]:

if msvamp_data.get('_needs_translation'):
    from transformers import AutoTokenizer as NTok, AutoModelForSeq2SeqLM
    import torch
    from tqdm.auto import tqdm

    print('Translating MSVAMP via NLLB (this takes ~20-40 min on T4)...')
    nllb_t = NTok.from_pretrained('facebook/nllb-200-distilled-600M')
    nllb_m = AutoModelForSeq2SeqLM.from_pretrained(
        'facebook/nllb-200-distilled-600M',
        torch_dtype=torch.float16 if not CFG['bf16'] else torch.bfloat16
    ).to(CFG['device'])
    nllb_m.eval()

    en_items = msvamp_data['en']
    del msvamp_data['_needs_translation']

    @torch.no_grad()
    def translate_batch(texts, tgt_lang_code, batch_size=8):
        nllb_t.src_lang = 'eng_Latn'
        forced_bos = nllb_t.lang_code_to_id[tgt_lang_code]
        out_texts = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = nllb_t(batch, return_tensors='pt', padding=True,
                         truncation=True, max_length=256).to(CFG['device'])
            gen = nllb_m.generate(**enc, forced_bos_token_id=forced_bos, max_new_tokens=256)
            out_texts.extend(nllb_t.batch_decode(gen, skip_special_tokens=True))
        return out_texts

    for lang in [l for l in CFG['languages'] if l != 'en']:
        if lang in msvamp_data:
            continue
        code = CFG['nllb_codes'][lang]
        questions = [item['question'] for item in en_items]
        translated_q = translate_batch(questions, code)
        msvamp_data[lang] = [
            {'question': tq, 'answer': item['answer'], 'translated': True}
            for tq, item in zip(translated_q, en_items)
        ]
        print(f'  Translated MSVAMP [{lang}]: {len(msvamp_data[lang])} items')

    with open(f'{DRIVE_ROOT}/datasets/msvamp.json', 'w') as f:
        json.dump(msvamp_data, f, ensure_ascii=False, indent=2)
    print('✅ Translated MSVAMP saved to Drive.')
else:
    print('MSVAMP translation not needed.')

MSVAMP translation not needed.


In [ ]:
import random
import json
import re

def build_synthetic_numglue(n_train=1700, n_test=530):
    """
    Synthetic arithmetic problems matching NumGLUE's style.
    Paper used ~1700 problems for preference estimation (tasks 1,4,8).
    """
    random.seed(42)

    # Template: (question_template, answer_function, task_id)
    templates = [
        # Task 1: single arithmetic operation
        ('There are {a} students in a class. {b} more join. How many students are there now?',
         lambda a,b: a+b, 1),
        ('A factory makes {a} widgets per day. How many does it make in {b} days?',
         lambda a,b: a*b, 1),
        ('A store has {a} items. They sell {b}. How many remain?',
         lambda a,b: a-b, 1),
        ('What is the sum of {a} and {b}?',
         lambda a,b: a+b, 1),
        ('Multiply {a} by {b}.',
         lambda a,b: a*b, 1),
        # Task 4: fill\u2011in\u2011blank (simple arithmetic)
        ('{a} divided by {b} equals what number?',
         lambda a,b: a//b if b!=0 else 0, 4),
        ('What number added to {a} gives {c}?', # Modified: changed lambda to use 'a' and 'c'
         lambda a,c: c-a, 4),
        ('{a} minus {b} equals?',
         lambda a,b: a-b, 4),
        # Task 8: multi\u2011step
        ('{a} students each buy {b} pencils and {c} pens. How many items total?',
         lambda a,b,c: a*(b+c), 8),
        ('A car travels {a} mph for {b} hours, then {c} mph for {d} hours. Total distance?',
         lambda a,b,c,d: a*b + c*d, 8),
        ('A rectangle has length {a} and width {b}. Perimeter?',
         lambda a,b: 2*(a+b), 8),
        ('If a number {a} is increased by {b} and then doubled, what is the result?',
         lambda a,b: 2*(a+b), 8),
    ]

    def make_item(idx, task_override=None):
        tmpl, fn, task = templates[idx % len(templates)]
        if task_override is not None:
            task = task_override


        placeholder_names = sorted(list(set(re.findall(r'\{([a-z])\}', tmpl))))


        values_for_placeholders = [random.randint(2, 20) for _ in range(len(placeholder_names))]

        fmt_dict = dict(zip(placeholder_names, values_for_placeholders))


        if 'remain' in tmpl or 'sell' in tmpl or 'minus' in tmpl:
            if 'a' in fmt_dict and 'b' in fmt_dict:
                fmt_dict['b'] = min(fmt_dict['b'], fmt_dict['a'])

        if 'divided' in tmpl or '÷' in tmpl:
            if 'a' in fmt_dict and 'b' in fmt_dict and fmt_dict['b'] != 0:
                fmt_dict['a'] = fmt_dict['a'] * fmt_dict['b']

        q = tmpl.format(**fmt_dict)


        fn_arg_names = fn.__code__.co_varnames[:fn.__code__.co_argcount]
        fn_args_to_pass = [fmt_dict[arg_name] for arg_name in fn_arg_names]

        ans = str(fn(*fn_args_to_pass))
        return {'question': q, 'answer': ans, 'task': task, 'synthetic': True}

    train = [make_item(i) for i in range(n_train)]
    test  = [make_item(i) for i in range(n_test)]
    return train, test

print('Loading MNumGLUESub (synthetic, matches paper scale)...')
mnumglue_train, mnumglue_test = build_synthetic_numglue()
print(f'  ✅ Synthetic: {len(mnumglue_train)} train, {len(mnumglue_test)} test')

# Save to Drive
import os
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
with open(f'{DRIVE_ROOT}/datasets/mnumglue_train.json', 'w') as f:
    json.dump(mnumglue_train, f, ensure_ascii=False, indent=2)
with open(f'{DRIVE_ROOT}/datasets/mnumglue_test.json', 'w') as f:
    json.dump(mnumglue_test, f, ensure_ascii=False, indent=2)
print('MNumGLUESub saved to Drive.')

Loading MNumGLUESub (synthetic, matches paper scale)...
  ✅ Synthetic: 1700 train, 530 test
MNumGLUESub saved to Drive.


---
## Section 3 — Load Base Model

In [ ]:

import os
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


TOKEN = os.environ.get('HF_TOKEN', None)

# 8-bit config with CPU offload for fp32 modules
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
    bnb_8bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

print(f'Loading: {CFG["base_model"]}')

tokenizer = AutoTokenizer.from_pretrained(
    CFG['base_model'],
    use_fast=True,
    padding_side='left',
    token=TOKEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Custom device map: limit GPU to 10GB, rest to CPU
device_map = {
    "model.embed_tokens": 0,
    "model.layers": "auto",
    "model.norm": 0,
    "lm_head": 0,
}

max_memory = {0: "10GiB", "cpu": "30GiB"}

policy_model = AutoModelForCausalLM.from_pretrained(
    CFG['base_model'],
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    trust_remote_code=True,
    token=TOKEN,
    low_cpu_mem_usage=True,
)
policy_model.eval()
print(f'✅ Loaded. Params: {sum(p.numel() for p in policy_model.parameters())/1e9:.2f}B')
print(f'   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading: kevinpro/MathOctopus-MAPO-DPO-7B


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Loaded. Params: 6.74B
   VRAM used: 7.01 GB


In [ ]:
# ────────────────────────────────────────────────────────────
# Prompt template & sampling utility
# ────────────────────────────────────────────────────────────
PROMPT_TEMPLATE = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{question}\n\n'
    '### Response: Let\'s think step by step.\n'
)

def build_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(question=question)

from typing import List

@torch.no_grad()
def sample_outputs(
    model, tokenizer, question: str,
    n: int = 8, temperature: float = 0.7, max_new_tokens: int = 512,
) -> List[str]:
    prompt = build_prompt(question)
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=1024
    ).to(model.device)
    outputs = model.generate(
        **inputs,
        do_sample=True,
        temperature=temperature,
        num_return_sequences=n,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
    )
    pl = inputs['input_ids'].shape[1]
    return [tokenizer.decode(o[pl:], skip_special_tokens=True) for o in outputs]

# Quick sanity inference
print('Testing inference...')
test_out = sample_outputs(policy_model, tokenizer, 'What is 15 + 27?', n=1)
print('Model output:', test_out[0][:200])

Testing inference...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Model output: 15 + 27 = <<15+27=42>>42
#### 42


---
## Section 4 — NLLB Translation Model

In [ ]:
# ────────────────────────────────────────────────────────────
# Load NLLB-200-distilled-600M with correct alignment scoring
# ────────────────────────────────────────────────────────────
from transformers import AutoTokenizer as NTok, AutoModelForSeq2SeqLM
import torch, math

print('Loading NLLB-200-distilled-600M...')
nllb_tokenizer = NTok.from_pretrained('facebook/nllb-200-distilled-600M')
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    'facebook/nllb-200-distilled-600M',
    torch_dtype=torch.bfloat16 if CFG['bf16'] else torch.float16,
).to(CFG['device'])
nllb_model.eval()
print('✅ NLLB loaded.')

@torch.no_grad()
def compute_alignment_score(non_en_text: str, en_text: str, src_lang: str) -> dict:
    """
    Compute PPL of English target given non-English source.
    Lower PPL = better alignment.
    """

    src_code = CFG['nllb_codes'].get(src_lang, src_lang)
    tgt_code = 'eng_Latn'


    nllb_tokenizer.src_lang = src_code

    nllb_tokenizer.tgt_lang = tgt_code


    inputs = nllb_tokenizer(
        non_en_text, return_tensors='pt', truncation=True, max_length=512
    ).to(CFG['device'])

    targets = nllb_tokenizer(
        text_target=en_text, return_tensors='pt', truncation=True, max_length=512
    ).to(CFG['device'])

    # Forward pass with labels
    outputs = nllb_model(
        input_ids=inputs['input_ids'],
        attention_mask=inputs.get('attention_mask', None),
        labels=targets['input_ids']
    )
    loss = outputs.loss.item()
    ppl = math.exp(min(loss, 20))
    return {'ppl': ppl, 'loss': loss}

s = compute_alignment_score(
    '建议加入土豆泥的学生人数是182。建议加入培根的学生人数是182 + 166 = 348。',
    'The number of students who suggested mashed potatoes is 182. '
    'The number who suggested bacon is 182+166=348.',
    src_lang='zh',
)
print(f'Paper Table 6 validation — PPL after alignment: {s["ppl"]:.2f}  (paper reports 0.97)')

s2 = compute_alignment_score(
    '建议加入土豆泥的学生比建议加入培根的学生多166人，所以差值为182 - 166 = 16。',
    'The number of students who suggested mashed potatoes is 182. '
    'The number who suggested bacon is 182+166=348.',
    src_lang='zh',
)
print(f'                         PPL before alignment: {s2["ppl"]:.2f}  (paper reports 2.65)')


Loading NLLB-200-distilled-600M...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

✅ NLLB loaded.
Paper Table 6 validation — PPL after alignment: 7.51  (paper reports 0.97)
                         PPL before alignment: 53.75  (paper reports 2.65)


---
## Section 5 — Preference Estimation

In [ ]:
# ────────────────────────────────────────────────────────────
#  Build preference dataset from MNumGLUESub
# ────────────────────────────────────────────────────────────
from tqdm.auto import tqdm
import json, os

def build_preference_dataset(
    model, tokenizer, questions_en, questions_non_en,
    n_samples=8, save_path=None,
):
    preference_data = []
    model.eval()

    for idx, en_item in enumerate(tqdm(questions_en, desc='Preference estimation')):
        q_en = en_item['question']
        en_samples = sample_outputs(model, tokenizer, q_en, n=1)
        y_bar = en_samples[0]

        for lang, lang_items in questions_non_en.items():
            if idx >= len(lang_items):
                continue
            q_src = lang_items[idx]['question']
            src_code = CFG['nllb_codes'].get(lang, 'eng_Latn')

            src_samples = sample_outputs(model, tokenizer, q_src, n=n_samples)

            scored = []
            for y_i in src_samples:
                try:
                    s = compute_alignment_score(y_i, y_bar, src_lang=src_code)
                    scored.append((y_i, s['ppl']))
                except Exception:
                    scored.append((y_i, 999.0))

            scored.sort(key=lambda x: x[1])   # lower PPL = better = chosen

            if len(scored) >= 2:
                preference_data.append({
                    'question_en':      q_en,
                    'question_src':     q_src,
                    'lang':             lang,
                    'y_bar':            y_bar,
                    'chosen':           scored[0][0],
                    'rejected':         scored[-1][0],
                    'score_chosen':     scored[0][1],
                    'score_rejected':   scored[-1][1],
                    'all_samples':      [{'text': t, 'ppl': p} for t, p in scored],
                })

    if save_path:
        with open(save_path, 'w') as f:
            json.dump(preference_data, f, ensure_ascii=False, indent=2)
        print(f'Saved: {save_path} ({len(preference_data)} pairs)')
    return preference_data

print('Preference dataset builder defined.')

Preference dataset builder defined.


In [ ]:
# ────────────────────────────────────────────────────────────
# Helper: Generate reasoning and extract answer
# ────────────────────────────────────────────────────────────
SYSTEM_PROMPT = "You are a helpful assistant that solves math problems step by step."

def generate_reasoning(model, tokenizer, question, max_new_tokens=512):
    """Generate step‑by‑step reasoning for a given question."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Solve this math problem step by step: {question}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=CFG['temperature'],
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

def extract_answer(text):
    """Extract numeric answer from reasoning text."""
    import re
    lines = text.strip().split('\n')
    for line in reversed(lines):
        line = line.lower()
        if 'answer is' in line or 'the answer' in line:
            nums = re.findall(r'\b(\d+(?:\.\d+)?)\b', line)
            if nums:
                return nums[-1]
    nums = re.findall(r'\b(\d+(?:\.\d+)?)\b', text)
    return nums[-1] if nums else None

In [ ]:
# ────────────────────────────────────────────────────────────
# 5.2  Translate MNumGLUESub to non-English languages
#      and run preference estimation
# ────────────────────────────────────────────────────────────
from tqdm import tqdm
import json, os, torch
import math

def build_preference_dataset(policy_model, tokenizer, questions_en, questions_non_en, n_samples=2, save_path=None):

    all_pref = []

    for lang, items_non_en in questions_non_en.items():
        print(f"\nProcessing language: {lang}")
        # For each problem, sample n_samples reasoning traces in both English and non-English
        for idx, (en_item, non_en_item) in enumerate(zip(questions_en, items_non_en)):
            # Skip if we already have enough
            if len(all_pref) >= 500 and save_path is not None:
                break

            en_q = en_item['question']
            non_en_q = non_en_item['question']
            correct_ans = en_item['answer']

            # Sample multiple reasoning traces
            en_reasonings = []
            non_en_reasonings = []

            for _ in range(n_samples):
                # English reasoning
                en_r = generate_reasoning(policy_model, tokenizer, en_q)
                en_reasonings.append(en_r)
                # Non-English reasoning
                non_en_r = generate_reasoning(policy_model, tokenizer, non_en_q)
                non_en_reasonings.append(non_en_r)

            en_ref = en_reasonings[0]
            scored = []
            for nr in non_en_reasonings:
                score = compute_alignment_score(nr, en_ref, lang)
                scored.append((nr, score['ppl']))


            scored.sort(key=lambda x: x[1])

            for i in range(len(scored)-1):
                all_pref.append({
                    'prompt': f"Solve this math problem step by step: {non_en_q}",
                    'chosen': scored[i][0],
                    'rejected': scored[i+1][0],
                    'lang': lang,
                    'ppl_chosen': scored[i][1],
                    'ppl_rejected': scored[i+1][1],
                })


            if len(all_pref) >= 200 and save_path is not None:
                break

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, 'w') as f:
            json.dump(all_pref, f, indent=2)
    return all_pref

# Cache path
PREF_CACHE = f'{DRIVE_ROOT}/preference_data/pref_iter1.json'

if os.path.exists(PREF_CACHE):
    print(f'Loading cached preference data...')
    with open(PREF_CACHE) as f:
        preference_data = json.load(f)
    print(f'✅ Loaded {len(preference_data)} preference pairs.')
else:

    FULL_RUN = False
    N_EXAMPLES = len(mnumglue_train) if FULL_RUN else 25
    print(f'Running preference estimation on {N_EXAMPLES} examples...')

    questions_en = [
        {'question': item['question'], 'answer': item['answer']}
        for item in mnumglue_train[:N_EXAMPLES]
    ]

    questions_non_en = {}
    for lang in [l for l in CFG['languages'] if l != 'en']:
        code = CFG['nllb_codes'][lang]
        translated = []
        for item in tqdm(questions_en, desc=f'Translating→{lang}', leave=False):
            # Set source language to English
            nllb_tokenizer.src_lang = 'eng_Latn'
            enc = nllb_tokenizer(item['question'], return_tensors='pt',
                                  truncation=True, max_length=256).to(CFG['device'])
            # Get target language token ID using convert_tokens_to_ids
            forced_id = nllb_tokenizer.convert_tokens_to_ids(code)
            with torch.no_grad():
                out = nllb_model.generate(**enc, forced_bos_token_id=forced_id, max_new_tokens=256)
            translated.append({
                'question': nllb_tokenizer.decode(out[0], skip_special_tokens=True),
                'answer': item['answer']
            })
        questions_non_en[lang] = translated
        print(f'  Translated {len(translated)} items to [{lang}]')
# After loading the tokenizer, set a simple chat template
    tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'user' %}[INST] {{ message['content'] }} [/INST]{% elif message['role'] == 'assistant' %}{{ message['content'] }}{% endif %}{% endfor %}"
    preference_data = build_preference_dataset(
        policy_model, tokenizer,
        questions_en, questions_non_en,
        n_samples= 3,
        save_path=PREF_CACHE,
    )

print(f'\nPreference pairs: {len(preference_data)}')

Running preference estimation on 25 examples...


  Translated 25 items to [bn]


  Translated 25 items to [th]


  Translated 25 items to [sw]


  Translated 25 items to [ja]


  Translated 25 items to [zh]


  Translated 25 items to [ru]


  Translated 25 items to [de]


  Translated 25 items to [es]


  Translated 25 items to [fr]

Processing language: bn

Processing language: th

Processing language: sw

Processing language: ja

Processing language: zh

Processing language: ru

Processing language: de

Processing language: es

Processing language: fr

Preference pairs: 210
